In [1]:
# Install the GitHub CardPy checkout into this notebook's Python environment.
from pathlib import Path
import subprocess
import sys

CARDPY_REPO = Path.home() / "Documents" / "GitHub" / "CarDpy"
if not (CARDPY_REPO / "pyproject.toml").is_file():
    raise FileNotFoundError(f"CardPy checkout not found: {CARDPY_REPO}")

# Equivalent to: cd ~/Documents/GitHub/CarDpy && python -m pip install -e .
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    cwd=str(CARDPY_REPO),
)


Obtaining file:///Users/ariel/Documents/GitHub/CarDpy
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
  Building editable for cardpy-cmr (pyproject.toml): started
  Building editable for cardpy-cmr (pyproject.toml): finished with status 'done'
  Created wheel for cardpy-cmr: filename=cardpy_cmr-0.1.1-0.editable-py3-none-any.whl size=29656 sha256=d9383e0af68e48c6d1b10a6bbaca9726d7349e1427ea7f0e336c6f989cdb1e75
  Stored in d

0

In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [3]:
from pathlib import Path
import os
import sys

# Use the current GitHub checkout rather than Scripts/CarDpy-master.
CARDPY_REPO = Path.home() / 'Documents' / 'GitHub' / 'CarDpy'
if not (CARDPY_REPO / 'cardpy' / '__init__.py').is_file():
    raise FileNotFoundError(f'CardPy checkout not found: {CARDPY_REPO}. Update CARDPY_REPO above.')

# A kernel that already imported legacy CardPy must be restarted first.
for module_name, module in tuple(sys.modules.items()):
    if module_name == 'cardpy' or module_name.startswith('cardpy.'):
        module_file = getattr(module, '__file__', None)
        if module_file and not Path(module_file).resolve().is_relative_to(CARDPY_REPO.resolve()):
            raise RuntimeError('A different CardPy copy is already loaded. Restart the kernel and run this notebook again.')
sys.path.insert(0, str(CARDPY_REPO))
import cardpy
print(f'Using CardPy: {Path(cardpy.__file__).resolve()}')
import glob
from smart_health_config import get_config
# Compatibility process preserves the original sequence and output numbering.
from smart_health_processing import process


Using CardPy: /Users/ariel/Documents/GitHub/CarDpy/cardpy/__init__.py


In [4]:
# Shared original settings live in smart_health_config.py.
config = get_config("blip")


In [ ]:
volunteer = 'SCH_Patient_07' # 'SCH_vol_XX or SCH_Patient_XX' 
distortion = 'BlipUp' # or 'Blip_Down'
sl_id = 'Even' # Even or Odd
even_odd_mode = False # True or False if acquired even/odd slices seperately on will handle the even/odd 

main_data_folder = '/Volumes/T7_old/Smart_Health/02_Anonymized_Data' # path to the main data folder where the volunteer folder is in
main_output_folder = '/Volumes/T7_old/Smart_Health/03_Processed_Data/02_cDTI/NifTi' # path to the main output folder where the volunteer folder is in

In [ ]:
study_root = os.path.join(main_data_folder, volunteer)
acquisition = distortion + sl_id if even_odd_mode else distortion
# Anonymized data use 02_cDTI; original data place the acquisition directly
# below the subject. Resolve against the selected root and check before import.
candidates = [
    os.path.join(study_root, '02_cDTI', acquisition),
    os.path.join(study_root, acquisition),
]
input_directory = next((path for path in candidates if os.path.isdir(path)), None)
if input_directory is None:
    raise FileNotFoundError('Blip Up input directory not found. Checked: ' + ', '.join(candidates))
config["dicom_subpath"] = os.path.relpath(input_directory, study_root)

out_path = os.path.join(main_output_folder, volunteer, 'Blip_Up')
if even_odd_mode:
    out_path += '_' + sl_id
config["output"]["directory"] = out_path


In [10]:
results = process(study_root, config)


Number of Diffusion Directions: 16
Number of Slices: 6
Number of Columns: 128
Number of Rows: 128
Number of Averages: 5
Gibbs ringing removal mode is on.
Rejection mode is on.
Re-registering data prior to data rejection.


/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Registering 2D data to lowest b-value (0).
Slice Crop Coordinates: [[42, 42, 42, 42, 42, 42], [84, 84, 83, 80, 80, 80], [39, 39, 39, 43, 43, 45], [90, 85, 85, 84, 84, 84]]


/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (5) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (5) found smaller than n_clusters (7). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (5) found smaller than n_clusters (8). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (5) found smaller than n_clusters (9). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **

Overridding estimated number of clusters to be 5 clusters.


/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (72) found smaller than n_clusters (73). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (72) found smaller than n_clusters (74). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (72) found smaller than n_clusters (75). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Overridding estimated number of clusters to be 5 clusters.
Overridding estimated number of clusters to be 5 clusters.
Initialization complete
Iteration 0, inertia 0.14841839195109885
Iteration 1, inertia 0.043618758723057496
Iteration 2, inertia 0.04125416823701625
Iteration 3, inertia 0.0407229330892361
Iteration 4, inertia 0.04062082934080692
Iteration 5, inertia 0.04049865846862668
Iteration 6, inertia 0.04043478493722648
Converged at iteration 6: strict convergence.
Overridding estimated number of clusters to be 5 clusters.
Initialization complete
Iteration 0, inertia 0.1257560873924829
Iteration 1, inertia 0.08527147793662879
Iteration 2, inertia 0.07253855429854389
Iteration 3, inertia 0.07148504589704803
Iteration 4, inertia 0.07056180074113275
Iteration 5, inertia 0.06971332695145377
Converged at iteration 5: strict convergence.
Initialization complete
Iteration 0, inertia 0.09849193518715815
Iteration 1, inertia 0.05775813600833493
Iteration 2, inertia 0.040981542305775026
Ite

/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (74) found smaller than n_clusters (75). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Overridding estimated number of clusters to be 5 clusters.
Initialization complete
Iteration 0, inertia 0.5213598595175176
Iteration 1, inertia 0.06833898304550742
Iteration 2, inertia 0.062329304308164496
Iteration 3, inertia 0.05941453878266159
Iteration 4, inertia 0.057378823638310913
Iteration 5, inertia 0.055788487050407884
Iteration 6, inertia 0.05562887623668922
Converged at iteration 6: strict convergence.
Acceptance rate for slice 1, direction 1: 100.0%
Acceptance rate for slice 1, direction 2: 100.0%
Acceptance rate for slice 1, direction 3: 100.0%
Acceptance rate for slice 1, direction 4: 100.0%
Acceptance rate for slice 1, direction 5: 100.0%
Acceptance rate for slice 1, direction 6: 100.0%
Acceptance rate for slice 1, direction 7: 100.0%
Acceptance rate for slice 1, direction 8: 100.0%
Acceptance rate for slice 1, direction 9: 100.0%
Acceptance rate for slice 1, direction 10: 100.0%
Acceptance rate for slice 1, direction 11: 100.0%
Acceptance rate for slice 1, direction 12